# Main file 

05.11.2025

Written by: Arnav Kumar Upadhyay, Cedric Meyer adapted from MATLAB code of Marc, Jan Spindler DOI-https://doi.org/10.1021/acssynbio.4c00578

This script is the main script for the TXTL v2 model.


In [1]:
# Packages functions and data types --------
include("packages.jl") 

# Sequences 
sequence_file	    = "Ecoli_K12_MG1655_geneset_cds_hpaBC_ddc.fa"    # "lacS_background.fa"
hybrid_file		    = "lacS.txt"
seqs_to_use 	    = (1,2323)                                 # Range of sequences in Fasta file. Takes all sequences from sequence a to sequence b
num_background_genes = 0
heterologous_genes  = ["hpaBC"]# ,"ddc"]

# initialization of data containers
scale = txtl_types.init_scale()         # struct that makes it possible to influence rates by scaling them

µ = 1                    # A growth rate (0.1 to 1.8) can be chosen and the proteins of the TXTL machinery will be tuned according to that growth rate and the processivity of the RNA polymerase

## flags for efficiency factors
positive_aminoacid	= true 
anti_shinedalgarno	= false      # If no hybridization file is available put flag to false. We apply the intaRNA program's intarna function (q: aSD, t: sequence, SeedBP: 5, out: tMinE) to generate a hybridization profile along the sequence. Link to the website: https://rna.informatik.uni-freiburg.de/IntaRNA/Input.jsp;jsessionid=2AE257E91E7276283F22CD77A5A8AA03
dynamic_efficiency	= true 
save_data           = false

flags = txtl_types.flags(positive_aminoacid, anti_shinedalgarno, dynamic_efficiency)

# Data import --------
mRNA 		= to_DataFrame(readmatrix("../data/input/mRNA_concentrations.xlsx", "mRNA_concentration", "A1:B2324"))
mRNA.mRNA 	= Float64.(mRNA.mRNA);

background_genes = String.(first(mRNA.gene, num_background_genes))
genes = vcat(background_genes, heterologous_genes)
println("Number of sequences: $(length(genes))") 

mRNA = subset(mRNA, :gene => ByRow(in(genes)))


## Import information about model sp ecies and sequences 
seqs, seq_names, hyb_file_dict, species_and_seqs, idx = species_and_sequences(genes, µ, sequence_file, hybrid_file, flags, seqs_to_use)
# Are mRNA gene names in E_coli gene 


## Calculate efficiency factors 
efac_pos_aa_dict, efac_int_sd_dict, dynamic_efficiencies = efficiency_factor_initialization(seqs, seq_names, hyb_file_dict, species_and_seqs)

## calculation of initial concentrations
y0, initial_concentrations = calculation_of_initial_concentration(species_and_seqs, seqs, seq_names, scale, idx);

## calculate and load kinetic parameters
p_met, p_seq, p_dyn = parameters(initial_concentrations, species_and_seqs);

## Release factor balance
RF_balance!(y0, initial_concentrations)

# Preintegration
## Definition of mass matrix
conserved = [idx.EFG, idx.EFD] # EFG, EFTu_GDP - Conserved species, Algebraic equations are given by mass balances

M = Matrix{Float64}(I, length(y0), length(y0)) # Creates unity matrix in the size of y0

M[conserved, :] .= 0; # setting mass matrix to zero to  remove time derivatives from those variables. This converts these differential equations into algebraic constraints


tags_pre = txtl_types.data_struct_preintegration(scale, idx, species_and_seqs, flags, dynamic_efficiencies, initial_concentrations, p_met, p_seq, p_dyn, mRNA.mRNA[1])

## Integration of the DAE system without DNA in the system 
y_steady_state = preintegration(tags_pre, y0, M);


# Elongation factor balance 
EF_balance!(tags_pre, y0, y_steady_state)


# Build states vector for multiple sequences 
y0_all = combined_states_vector(tags_pre, y_steady_state, seqs, seq_names)

# include_mRNA_concentrations!(y0_all, tags_pre, seq_names, seqs, mRNA)

# Prepare per-sequence tags (for use in ODE)
local_tags_list = generate_local_tags(seqs, seq_names, tags_pre, efac_pos_aa_dict, efac_int_sd_dict, mRNA);

# Integration
M = Matrix{Float64}(I, length(y0_all), length(y0_all))

M[conserved, :] .= 0

tspan 	= (0.0, 100.0) ## Define time span for the simulation

p_parallel = local_tags_list

f 		= ODEFunction(ODE_parallel_distributed!, mass_matrix = M) ## Define ODEFunction to make use of mass matrix

prob 	= ODEProblem(f, y0_all, tspan, p_parallel); ## Create an ODE problem

sol = solve(prob, FBDF(autodiff=false), reltol = 1e-8, abstol = 1e-6); ## Solve ODE system


🔧 Julia started with 4 threads
Number of sequences: 1


In [4]:
# Inputs to test functions
du = similar(y0_all)
du = fill!(du, 0.0);
u = sol[end];
t = tspan;

@save "initial_values_for_code_optimization.jld2" du u t p_parallel

In [ ]:

# function ODE_parallel_distributed!(du, u, p, t)

    local_tags_list = p_parallel				# Extraction of local_tags_list from p
    tags            = local_tags_list[1]
    n_metab = length(tags.species_and_sequences.Tag) -4 		# number of metabolites in the shared pool that is used by all sequences 
    @views y_metab =  u[1:n_metab]				# Save all pool species from u in a vector 
    idx = n_metab						# Index of last pool species


	y_codons_lengths = Int[length(local_tags_list[i].species_and_sequences.Seq) + 3 for i in eachindex(local_tags_list)]	# List with sequence lengths - 1 stop codon + 5 observable variables


	y_codons = Vector{SubArray{Float64,1,typeof(u),Tuple{UnitRange{Int64}},true}}(undef, length(y_codons_lengths))
    idx = n_metab
    @views for (i,L) in enumerate(y_codons_lengths)
        y_codons[i] = u[idx+1:idx+L]  # ✅ View statt Kopie
        idx += L
    end

    
    n_seq = tags.species_and_sequences.no_of_seqs 		# save number of sequences
    du_metab_all = [zeros(n_metab) for _ in 1:n_seq]	# List to store a du vector for the metabolite pool for every sequence 
    du_codons = Vector{Vector{Float64}}(undef, n_seq)	# List to store a du vector for the codons for every sequence 

    # THREADED parallelization of per-sequence rate calculations
    # ONLY CHANGE: map -> Threads.@threads for
    results = Vector{Vector{Float64}}(undef, n_seq)  # Pre-allocate results array

    i = 1
 
    local_tags = local_tags_list[i]		# Extraction of local tags
    u_seq = vcat(y_metab, y_codons[i])				# Sequenzspezifischer state vector
    du_seq = zeros(length(u_seq))					# Sequenzspezifischer rates vector

    fill!(du, 0.0)  # Reset to zero, just like initializing with zeros

    



    rhs_metabolites!(du, u, local_tags)
    

    # @benchmark rhs_sequence!(du, u, local_tags)
    rhs_sequence!(du, u, local_tags)

    # for i in du println(i) end 
    
    


In [2]:
for i in sol[end] println(i) end

20.065442041776336
20.08441495245314
0.011765402229386797
0.020943145550994333
2.3889999533484603
1.5446150070394027
0.6637306843760689
0.7463818107395602
0.21061899830763406
2.361917500157673
0.09469119446197172
24.154502728924523
15.794527928383367
23.67831171821118
33.05785547104019
1.2289697285377902e-6
0.40998532692557077
0.2365040717734266
3.36426489847972
0.0005784628892925569
0.000264006281126925
0.0001528894125371132
0.00013205929110925797
2.6705373411897394e-5
0.0003436721057947688
1.6729730903295083e-5
0.0001199403397876133
3.776832901533287e-5
0.0002684535448215428
0.0002773058325360847
0.0003114880876779609
0.00010355928245686178
5.9623815113732094e-5
1.7022579562001503e-5
4.3385436903642536e-5
0.0010450235036755047
2.296731002482089e-5
0.00014615656589415292
1.6787376327159956e-5
0.0002400640349932828
1.72369271185932e-5
3.27518623994609e-5
0.0012544102511604588
1.2882923484567426e-20
1.6182817370496535e-20
1.0155619458549936e-20
3.503674443901486e-5
4.077451739301873e-6


In [3]:

# du = zeros(Float64, length(u))
tags = local_tags
# Unpack parameters 
@unpack µ, k_proc, k_cat_Pol, C_RNP, V_TCmax, K_M_D, k_cat_E, C_RNE, V_degmax, K_M_deg, k_D, n_DNA_t, n_DNA_s = tags.p_seq.TX
@unpack k_TLEmaxj, KM_T3j, KM_EFGGTP = tags.p_seq.EL
@unpack k_TE, KM_RK, KM_GTP, KM_RF3GTP, k_RF3_diss_GDP, k_RF1_diss_GTP, k_GTP_hydr, k_RF1_ass = tags.p_seq.TER
@unpack k_RF3_GDP_ass, k_RF3_GTP_ass, k_RF3_GDP_diss, k_RF3_GTP_diss = tags.p_seq.RF3
@unpack A1, A2, A3, A4, A5, k_cat_K_M_G1, k_cat_K_M_G2, k_RRF, q_RRF, one_q_G2_k_split = tags.p_seq.PTC 

# Unpack constant variables 
ATP, GTP, UTP, CTP, ADP, GDP, UDP, CDP, AMP, GMP, UMP, CMP, Eut, Est, RFt, RRF, Ala, Cys, 
Asp, Glu, Phe, Gly, His, Ile, Lys, Leu, Met, Asn, Pro, Gln, Arg, Ser, Thr, val, Trp, Tyr, C_D = tags.init_c.constant_vars 

# Unpack dynamic variables 
S30, S50, IF1, IF2, IF3, SC1, SC2, SC3, SC4, PIC, EFG, EFD, EGT, EGD, EFT, RF3, RGT, RGD, b1t, ta1, ta2, tc1, td1, te2, tf1, tg2, tg3, th1, ti1, tk1, t1l, t2l, t3l, t4l, t5l, tm1, tn1, tp1, tp2, tp3, 
tq1, tq2, tr2, tr3, tr4, tr5, ts1, ts2, ts3, ts5, tt1, tt2, tt4, tv1, tv2, tw1, ty1, tA1, tA2, tC1, tD1, tE2, tF1, tG2, tG3, tH1, tI1, tK1, tL1, tL2, tL3, tL4, tL5, tM1, tN1, tP1, tP2, tP3, tQ1, tQ2, 
tR2, tR3, tR4, tR5, tS1, tS2, tS3, tS5, tT1, tT2, tT4, tV1, tV2, tW1, tY1, a1t, a2t, c1t, d1t, e2t, f1t, g2t, g3t, h1t, i1t, k1t, l1t, l2t, l3t, l4t, l5t, m1t, n1t, p1t, p2t, p3t, q1t, q2t, r2t, r3t, 
r4t, r5t, s1t, s2t, s3t, s5t, t1t, t2t, t4t, v1t, v2t, w1t, y1t, S70, rna, PRT, PTC = u 

# Containers 
num_species = length(tags.species_and_sequences.Name)
len_sequence = length(tags.species_and_sequences.Seq) # Codon length with stop codon

# Define RIB1

rate_RIB        = zeros(Float64, len_sequence-1);
@views c_RIB	= u[tags.idx_first_codon:end]

 

@views total_RNA   = tags.RNA + sum(c_RIB[1:12])

q_R_elo     = ones(len_sequence-1);
    
@views for i in 2:(len_sequence - 13)

    if total_RNA-sum(c_RIB[i:i+12]) == 0

        q_R_elo[i] 	= 0;
    else
        q_R_elo[i] 	= 1 - c_RIB[i+12] / (total_RNA-sum(c_RIB[i:i+11]));
    
    end
end 


@views for i in (len_sequence-13) : (len_sequence-1)
    if total_RNA - sum(c_RIB[i : end]) == 0
        q_R_elo[i] = 0
    else 
        q_R_elo[i] = 1 - (c_RIB[end] + PTC) / (total_RNA - sum(c_RIB[i:end-1]))
    end
end

# dynamic efficiency factor 
if tags.flags.dynamic_efficiency == true

    efac = dynEffficiencyCalculation(tags, EGT, u);

end
    

# V_EL
C_vect = reshape(u[1:num_species], 1, num_species)  # Directly assign
c_Codon = tags.t3_Codon_struc * C_vect'

V_TL_EL	= zeros(len_sequence-1);

EGT_inv = inv(EGT)


@inbounds for i in 1:len_sequence-2

    cod_next = tags.species_and_sequences.Seq[i+1]
    cod_curr = tags.species_and_sequences.Seq[i]

    j = tags.species_and_sequences.cod_idx[cod_next]    # O(1) Lookup
    k = tags.species_and_sequences.cod_idx[cod_curr]    # O(1) Lookup

    cc = c_Codon[j]

    if cc == 0
        V_TL_EL[i+1] = 0
    else
        eff = efac[k] * min(tags.efac_pos_aa[i], tags.efac_int_sd[i])

        denom = 1 + KM_T3j/cc + KM_EFGGTP*EGT_inv

        V_TL_EL[i+1] =
            q_R_elo[i] * k_TLEmaxj * c_RIB[i] * eff / denom
    end
end

V_TL_EL_sum = sum(V_TL_EL);

c_codon_seq = tags.matrix_N * c_Codon

N = C_vect[1,:] * (1 ./c_codon_seq)'

    # Replace Inf values with 0
    @inbounds for i in eachindex(N)
        if isinf(N[i])
            N[i] = 0
        end
    end

    # Replace NaN values with 0
    @inbounds for i in eachindex(N)
        if isnan(N[i])
            N[i] = 0
        end
    end

    M = zeros(size(N))

    for i in 1:size(N,1)
        for k in 1:size(N,2)
            M[i,k] = tags.alpha_struc[i,k] * N[i,k]
        end
    end

    K_T3 = M*V_TL_EL;


    mtRNA 		= zeros(size(M,1),2);
    M_tRNA 		= hcat(mtRNA,M)
    M_tRNA		= M_tRNA[:, Not(3)]
    M_tRNA		= M_tRNA[:, 1:end-1]

    K_tRNA 		= M_tRNA*V_TL_EL   


## translation termination       
    ## translation termination
    te = len_sequence-1 # Number of last coding codon
    # ke = findfirst(x -> x == tags["Seq1"]["Codon"][te], tags["cod"])

    k_RF1_diss_GTP = k_RF1_diss_GTP#*tags.scale.termination

    efficiency_term = 1;
    k_ter_RF1 = 1 / ( 1/(k_RF1_ass*RFt) + 1/(k_RF1_diss_GTP) );
    k_ter_RF3 = 1 / ( 1/(k_RF1_ass*RGT) + 1/(k_RF3_diss_GDP) + 1/(k_GTP_hydr) );

    if k_ter_RF1 < k_ter_RF3
        k_ter = k_ter_RF1;
    else
        k_ter = k_ter_RF3;
    end


    V_TE_max = k_ter * RFt   # RFt is RF1/2 total
    V_TL_TE = V_TE_max * efficiency_term / (1 + KM_RK/ c_RIB[te] + KM_RF3GTP/RGT + KM_RK * KM_RF3GTP /(c_RIB[te] * RGT))
        
    if c_RIB[te] == 0 || GTP == 0
        V_TL_TE=0;
    end

    V_RF3_GDP_ass  = k_RF3_GDP_ass  * RF3 * GDP;
    V_RF3_GTP_ass  = k_RF3_GTP_ass  * RF3 * GTP;
    V_RF3_GDP_diss = k_RF3_GDP_diss * RGD;
    V_RF3_GTP_diss = k_RF3_GTP_diss * RGT 

    ## post termination complex recycling
    t_rec = A1 + A2 * 1/RRF + A3 * EGT/RRF + A4 * 1/EGT + A5 * 1/(RRF*EGT); # time/s for recycling according to Borg et al. 2016
    k_rec = 1/t_rec; # rate/s^-1 for recycling according to Borg et al. 2016
    k_rec_min = k_rec*60; # rate/min^-1 for recycling according to Borg et al. 2016
    f_s = k_cat_K_M_G1/(k_RRF * RRF)*(q_RRF/k_cat_K_M_G2*one_q_G2_k_split + EGT)+ one_q_G2_k_split; #EGT hydrolised per splitting event according to Borg et al 2016

    # translation termination - calculation of du 

    @inbounds for i in 1:num_species
        du[i] += tags.alpha_struc[i, end] * V_TL_TE
    end

    rate_RIB[len_sequence-1] -= V_TL_TE


    du[tags.idx.PTC] += V_TL_TE - k_rec_min * PTC;	            # PTC
    du[tags.idx.S30] += k_rec_min * PTC;                         # rate of 30S release due to recycling
    du[tags.idx.S50] +=  k_rec_min * PTC;                        # rate of 50S release due to recycling
    du[tags.idx.PRT] = V_TL_TE;                                  # rate of PRT (Protein)
    du[tags.idx.RGT] = -V_TL_TE -V_RF3_GTP_diss +V_RF3_GTP_ass;  # rate of RGT
    du[tags.idx.RGD] =  V_TL_TE -V_RF3_GDP_diss +V_RF3_GDP_ass;  # rate of RGD
    du[tags.idx.RF3] =          +V_RF3_GDP_diss +V_RF3_GTP_diss -V_RF3_GTP_ass -V_RF3_GDP_ass; # rate of RF3


    # translation elongation - calculation of du 


    # concentration of the ribosome at the specific codon
    rate_RIB[1] = - V_TL_EL[2];  

    for i in 2: len_sequence-2
        rate_RIB[i] = V_TL_EL[i] - V_TL_EL[i+1];	
    end

    rate_RIB[len_sequence-1] = rate_RIB[len_sequence-1] + V_TL_EL[end];

    # concentration of the unloaded tRNA
    du[tags.idx.ta1:tags.idx.ty1] .+= K_tRNA[end-tags.species_and_sequences.no_of_tRNAs+1:end]

    # concentration of the ternary-complex
    du[1: num_species] .+= - K_T3;
    du[tags.idx.EGT]  += - V_TL_EL_sum - f_s * k_rec_min * PTC;	# Rate of EFG_GTP 
    du[tags.idx.EGD]  += + V_TL_EL_sum + f_s * k_rec_min * PTC;	# Rate of EFG_GDP
    # du[tags.idx.RNA] += + V_TL_EL[12];
    du[tags.idx.rna] += - V_TL_EL[12];

    # save RIB in du
    du[tags.idx_first_codon:end] .+= rate_RIB;

In [4]:
for i in du println(i) end

-1.6459271468221952e-8
-1.5371781897604957e-8
-2.180446036399719e-12
-1.4877729181007188e-10
-1.0874895108774554e-9
1.0720668797148392e-9
-1.334115040663164e-10
1.319211406780596e-11
3.418654248577013e-11
1.013983619291281e-10
0.0
3.126388037344441e-13
-2.1696263700521357e-5
2.1696351012012727e-5
4.615365534199611e-9
1.8673951274195133e-13
-1.8883383745560423e-10
1.8862023054566635e-10
-1.013983021895884e-10
5.1944619971558414e-6
2.588796375185831e-9
-1.5527169215309722e-8
3.588183772329501e-8
3.803177837970184e-6
1.2138261631933212e-7
4.015991823813625e-8
-4.9619642908393935e-9
7.250028963012856e-6
1.789250614413207e-6
1.9396751551811753e-7
2.456839387210419e-7
1.6560268896303326e-6
3.8405414033021446e-8
-5.761835124686379e-9
1.9593619608737622e-9
5.153749660299911e-7
-1.2235028407081572e-8
9.296198432220959e-8
4.165741503917714e-11
6.696948800545555e-8
2.2501912885081188e-9
-2.890199910302038e-8
3.4879602429926937e-7
-3.5176947954641603e-19
-4.418734032561406e-19
-2.7730017768754285e